# **NHS RTT Waiting Times Dashboard**

## Objectives

* The purpose of this dashboard is to inform a non-technical audience of NHS waiting times for various specialties.  The data includes the 25%, 50% and 75% quartiles for specialty waiting times.

## Inputs

* 20260831-RTT-August-2026-full-extract.csv

## Outputs

* To be updated by the end of the notebook

## Additional Comments

* Original input data downloaded as a ZIP file from https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/rtt-data-2026-27/#Aug26.
* File: https://www.england.nhs.uk/statistics/wp-content/uploads/sites/2/2026/10/Full-CSV-data-file-Aug26-ZIP-3M-vjgRdG.zip



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Code Institute - Data Analytics\\nhs-rtt-waiting-times-dashboard\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Code Institute - Data Analytics\\nhs-rtt-waiting-times-dashboard'

# Section 1

Section 1 content

In [ ]:
# Imports
import pandas as pd
 
# Load

df = pd.read_csv("data/raw/20260831-RTT-August-2026-full-extract.csv")
print(df.shape)

# Cleaning

 # Keep Part_2 rows i.e. 'Still waiting'
selection = df["RTT Part Type"] == "Part_2"
part2 = df[selection].copy()
# Remove 'Total' specialty rows to avoid double-counting
selection2 = part2["Treatment Function Code"] != "C_999"
part2 = part2[selection2].copy()

print(part2.shape)

# Check totals two ways:
# - see Markdown cell below

sum_a = part2["Total All"].sum()
print("sum_a =", sum_a)

selection3 = (df["RTT Part Type"] == "Part_2") & (df["Treatment Function Code"] == "C_999")
sum_b = df[selection3]["Total All"].sum()
print("sum_b =", sum_b)

assert sum_a == sum_b

(177770, 121)
(50549, 121)
sum_a = 7189213
sum_b = 7189213


Check totals two ways:

The checks validate the totals in two ways: either summing the specialty rows, or summing summary rows that already combine all specialties for each provider and commissioner, in each case for Part_2 rows i.e. patients currently on the waiting list.

- total 1 i.e. sum_a:
  - isolate Part_2 rows in a DataFrame.
  - exclude Total rows from DataFrame, leaving specialty data, and sum Total All.

- total 2 i.e. sum_b:
  - select Total rows (C_999) from the original df DataFrame (because they were dropped from the part2 DataFrame), and sum Total All.

The check confirms the filter removed double-counting without losing patients.  Both totals equal 7,189,213 patients waiting in England in August 2026.  No patients were lost when rows were excluded during cleaning.


## Cases Where Commissioner Parent is Blank

In [ ]:
blanks = part2["Commissioner Parent Name"].isna()
patients_blank = part2[blanks]["Total All"].sum()
print("Patients blank:", patients_blank)
share = patients_blank / sum_a * 100
print("Blanks as share of all patients:", share)


Patients blank: 161221
Blanks as share of all rows: 2.2425403169999276


## Comparing Like With Like

In [ ]:
# share of rows
average = part2["Commissioner Parent Name"].isna().mean() * 100
print(average)

5.992205582701933


In [16]:
blank_rows = part2[blanks]
by_name = blank_rows.groupby("Commissioner Org Name")["Total All"].sum()
by_name = by_name.sort_values(ascending=False)
by_name.head(10)


Commissioner Org Name
NORTH WEST COMMISSIONING HUB                        48370
NORTH EAST AND YORKSHIRE COMMISSIONING HUB          25269
SOUTH EAST COMMISSIONING HUB                        19973
SOUTH WEST COMMISSIONING HUB                        12489
NATIONAL COMMISSIONING HUB 1                         5986
NORTH EAST AND YORKSHIRE - H&J COMMISSIONING HUB     2518
MIDLANDS - H&J COMMISSIONING HUB                     1930
LONDON COMMISSIONING HUB                             1401
SOUTH EAST - H&J COMMISSIONING HUB                   1261
EAST OF ENGLAND - H&J COMMISSIONING HUB              1115
Name: Total All, dtype: int64

## Assigning Patients to a Commissioner Category

The columns in the waiting list were not uniform in what they represented and a new column was introduced to make the data easier to understand and work with.  Rows are classified into three groups as follows:
1. Commissioner Parent Name is present,
2. Commissioner Parent Name is absent, but Commissioner Org Name is present,
3. both Commissioner Parent Name and Commissioner Org Name are absent.  In these cases, there is always a Commissioner Org Code.

In [ ]:
# Create a column of commissioner information, with fallback when initial
# columns searched are empty, until a non-empty column is found.

# The column amalgamates data from other columns to indicate the commissioning
# group that generated the data as best as possible.

part2["commissioner_group"] = (
    part2["Commissioner Parent Name"]
    .fillna(part2["Commissioner Org Name"]
            .fillna(part2["Commissioner Org Code"])
            )
)

In [22]:
# Checks:
print(part2["commissioner_group"].isna().sum())
print(part2["Total All"].sum())
print(part2["commissioner_group"].nunique())

0
7189213
59


In [24]:
part2[["commissioner_group", "Treatment Function Name", "Total All"]].head(10)

,commissioner_group,Treatment Function Name,Total All
4,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,65
10,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,1
18,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,14
28,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,14
36,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,6
42,NHS ESSEX INTEGRATED CARE BOARD,Ophthalmology Service,3
48,NHS KENT AND MEDWAY INTEGRATED CARE BOARD,Ophthalmology Service,1
61,NHS ESSEX INTEGRATED CARE BOARD,Trauma and Orthopaedic Service,111
62,NHS ESSEX INTEGRATED CARE BOARD,Ear Nose and Throat Service,32
63,NHS ESSEX INTEGRATED CARE BOARD,Rheumatology Service,1


In [40]:
week_cols = [col for col in part2.columns if col.startswith("Gt ")]
print(len(week_cols))
print(week_cols[:3])
print(week_cols[-1])

105
['Gt 00 To 01 Weeks SUM 1', 'Gt 01 To 02 Weeks SUM 1', 'Gt 02 To 03 Weeks SUM 1']
Gt 104 Weeks SUM 1


In [48]:
group_cols = ["commissioner_group", "Treatment Function Code", "Treatment Function Name"]
value_cols = week_cols + ["Total All"] + ["Patients with unknown clock start date"]

len(group_cols), len(value_cols)

(3, 107)

In [50]:
icb_spec = part2.groupby(group_cols)[value_cols].sum().reset_index()
print(icb_spec.shape)
print(icb_spec["Total All"].sum())
icb_spec.head()

(1181, 110)
7189213


,commissioner_group,Treatment Function Code,Treatment Function Name,Gt 00 To 01 Weeks SUM 1,Gt 01 To 02 Weeks SUM 1,Gt 02 To 03 Weeks SUM 1,Gt 03 To 04 Weeks SUM 1,Gt 04 To 05 Weeks SUM 1,Gt 05 To 06 Weeks SUM 1,Gt 06 To 07 Weeks SUM 1,...,Gt 97 To 98 Weeks SUM 1,Gt 98 To 99 Weeks SUM 1,Gt 99 To 100 Weeks SUM 1,Gt 100 To 101 Weeks SUM 1,Gt 101 To 102 Weeks SUM 1,Gt 102 To 103 Weeks SUM 1,Gt 103 To 104 Weeks SUM 1,Gt 104 Weeks SUM 1,Total All,Patients with unknown clock start date
0,EAST OF ENGLAND - H&J COMMISSIONING HUB,C_100,General Surgery Service,3.0,4.0,5.0,5.0,4.0,4.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,132,0.0
1,EAST OF ENGLAND - H&J COMMISSIONING HUB,C_101,Urology Service,4.0,3.0,4.0,5.0,6.0,3.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,111,0.0
2,EAST OF ENGLAND - H&J COMMISSIONING HUB,C_110,Trauma and Orthopaedic Service,2.0,5.0,3.0,9.0,5.0,7.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,190,0.0
3,EAST OF ENGLAND - H&J COMMISSIONING HUB,C_120,Ear Nose and Throat Service,3.0,3.0,6.0,4.0,4.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,122,0.0
4,EAST OF ENGLAND - H&J COMMISSIONING HUB,C_130,Ophthalmology Service,1.0,4.0,0.0,4.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,75,0.0


In [ ]:
# Check each group-and-specialty combination appears only once
icb_spec.duplicated(subset=group_cols).sum()

0

In [75]:
icb_spec[week_cols].sum().sum()

7189213.0

In [81]:
icb_spec["Total All"].sum()

7189213

In [87]:
icb_spec["Patients with unknown clock start date"].sum()

0.0

All 7,189,213 patients in the Part_2 specialty rows have a known clock start date, so all contribute to the percentiles.

In [94]:
icb_spec.groupby("commissioner_group")["Total All"].sum().sort_values(ascending=False).head(3)

commissioner_group
NHS WEST AND NORTH LONDON INTEGRATED CARE BOARD    490660
NHS SURREY AND SUSSEX INTEGRATED CARE BOARD        414102
NHS GREATER MANCHESTER INTEGRATED CARE BOARD       411464
Name: Total All, dtype: int64

In [95]:
# Check grouping adds up correctly at the cell level
mask_a = (part2["commissioner_group"] == "NHS WEST AND NORTH LONDON INTEGRATED CARE BOARD") & (part2["Treatment Function Code"] == "C_110")
check_a = part2[mask_a]["Total All"].sum()
print(check_a)

38478


In [96]:
mask_b = (icb_spec["commissioner_group"] == "NHS WEST AND NORTH LONDON INTEGRATED CARE BOARD") & (icb_spec["Treatment Function Code"] == "C_110")
check_b = icb_spec[mask_b]["Total All"].sum()
print(check_b)

38478


In [ ]:
check_a, check_b

(38478, 38478)

In [ ]:
# Save ICB specialties file
icb_spec.to_csv("data/processed/v1/icb_specialty_waits.csv", index=False)

In [100]:
# Reload iCB specialties file and check
check = pd.read_csv("data/processed/v1/icb_specialty_waits.csv")
check.shape

(1181, 110)

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [101]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)